In [ ]:
import pandas as pd
import asyncio
import aiohttp
import os

In [24]:
ids = pd.read_json('data/movie_ids_07_27_2026.json', lines=True)
top_50k_ids = ids.nlargest(50000, 'popularity')['id'].tolist()

with open('data/top_50k_ids.txt', 'w') as f:
    for movie_id in top_50k_ids:
        f.write(f"{movie_id}\n")

In [ ]:
API_KEY = ''
BASE_URL = 'https://api.themoviedb.org/3/movie/'

with open('data/top_50k_ids.txt', 'r') as f:
    movie_ids = [line.strip() for line in f if line.strip()]

async def fetch_movie(session, movie_id, semaphore):
    async with semaphore:
        url = f"{BASE_URL}{movie_id}?api_key={API_KEY}&language=ru-RU&append_to_response=videos"
        
        for attempt in range(3):
            try:
                async with session.get(url) as response:
                    if response.status == 200:
                        data = await response.json()
                        
                        trailer_url = None
                        if 'videos' in data and 'results' in data['videos']:
                            for video in data['videos']['results']:
                                if video.get('type') == 'Trailer' and video.get('site') == 'YouTube':
                                    trailer_url = f"https://www.youtube.com/watch?v={video.get('key')}"
                                    break 
                        return {
                            'id': data.get('id'),
                            'title': data.get('title'),
                            'genres': [g['name'] for g in data.get('genres', [])],
                            'overview': data.get('overview'),
                            'poster_path': data.get('poster_path'),
                            'popularity': data.get('popularity'),
                            'vote_average': data.get('vote_average'),
                            'trailer_url': trailer_url
                        }
                    elif response.status == 429:
                        await asyncio.sleep(2 ** attempt)
                        continue
                    else:
                        return None
            except Exception as e:
                await asyncio.sleep(1)
        
        return None

import os

async def main():
    semaphore = asyncio.Semaphore(10)
    
    timeout = aiohttp.ClientTimeout(total=30) 
    
    csv_filename = 'tmdb_movies_ru.csv'

    batch_size = 1000
    batches = [movie_ids[i:i + batch_size] for i in range(0, len(movie_ids), batch_size)]
    
    async with aiohttp.ClientSession(timeout=timeout) as session:
        for index, batch in enumerate(batches):
            print(f"Скачиваем батч {index + 1} из {len(batches)}...")
            
            tasks = [fetch_movie(session, mid, semaphore) for mid in batch]
            results = await asyncio.gather(*tasks)

            valid_movies = [m for m in results if m is not None and m.get('overview')]
            
            if valid_movies:
                df = pd.DataFrame(valid_movies)

                if not os.path.isfile(csv_filename):
                    df.to_csv(csv_filename, index=False)
                else:
                    df.to_csv(csv_filename, mode='a', header=False, index=False)
                    
            print(f"Батч {index + 1} сохранен! Найдено {len(valid_movies)} фильмов.")

            await asyncio.sleep(2)
    
await main()

Скачиваем батч 1 из 50...
Батч 1 сохранен! Найдено 856 фильмов.
Скачиваем батч 2 из 50...
Батч 2 сохранен! Найдено 853 фильмов.
Скачиваем батч 3 из 50...
Батч 3 сохранен! Найдено 806 фильмов.
Скачиваем батч 4 из 50...
Батч 4 сохранен! Найдено 729 фильмов.
Скачиваем батч 5 из 50...
Батч 5 сохранен! Найдено 664 фильмов.
Скачиваем батч 6 из 50...
Батч 6 сохранен! Найдено 611 фильмов.
Скачиваем батч 7 из 50...
Батч 7 сохранен! Найдено 558 фильмов.
Скачиваем батч 8 из 50...
Батч 8 сохранен! Найдено 534 фильмов.
Скачиваем батч 9 из 50...
Батч 9 сохранен! Найдено 469 фильмов.
Скачиваем батч 10 из 50...
Батч 10 сохранен! Найдено 461 фильмов.
Скачиваем батч 11 из 50...
Батч 11 сохранен! Найдено 422 фильмов.
Скачиваем батч 12 из 50...
Батч 12 сохранен! Найдено 431 фильмов.
Скачиваем батч 13 из 50...
Батч 13 сохранен! Найдено 379 фильмов.
Скачиваем батч 14 из 50...
Батч 14 сохранен! Найдено 412 фильмов.
Скачиваем батч 15 из 50...
Батч 15 сохранен! Найдено 385 фильмов.
Скачиваем батч 16 из 50...
Б